# Séance 12 · Exercices — Comment fonctionne un LLM · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

**Niveau de la séance : ⭐⭐** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

Comment travailler : lis l'énoncé, code dans la cellule « À toi », lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
Tout tourne dans **Google Colab**, rien à installer. Exécute chaque cellule avec `Maj + Entrée`.
Tous les exercices marchent aussi en **mode démo** (`USE_MODEL = False`) : les réponses du modèle sont alors écrites à la main, mais les tokens, le bigramme et le sac de mots sont bien réels.


## Quiz d'ouverture (2 min)

Réponds dans ta tête, puis déplie la réponse. Ce sont les questions qu'on se pose à voix haute au début de la séance.

**1. Au fond, que fait un LLM quand il « écrit » ?**
- a. Il cherche la phrase sur internet
- b. Il prédit le token suivant, encore et encore
- c. Il réfléchit comme un humain
- d. Il pioche dans une liste de réponses toutes faites

<details><summary>Réponse</summary>

**b. Il prédit le token suivant, encore et encore** — Un token à la fois, en choisissant parmi les plus probables. C'est tout, et c'est ce qui rend les hallucinations possibles.

</details>

**2. Un modèle te donne le vainqueur de la coupe du monde 2031 avec assurance. C'est…**
- a. Une prédiction fiable
- b. Une hallucination
- c. Un bug
- d. Une blague du modèle

<details><summary>Réponse</summary>

**b. Une hallucination** — L'événement n'a pas eu lieu. Le modèle a produit des mots plausibles. Toujours vérifier une réponse qui compte.

</details>


## Préparation

La même cellule qu'à la leçon (elle prépare `llm(messages)`), plus le découpeur `tiktoken`, et des données neuves : les 30 phrases d'une pizzeria pour le mini-modèle, 20 avis de joueurs sur un jeu vidéo (positif / négatif), et le helper `verifier`. Lance-la une fois.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Réponses écrites à la main, choisies selon les mots de la question (mode démo)."""
    q = messages[-1]["content"].lower()
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system").lower()
    if "avis" in systeme:                      # exercice 11 : le modèle classe un avis de joueur
        negatif = ["déchant", "bug", "plante", "cher", "nul", "ennuy", "regrette", "mais", "pas "]
        return "négatif" if any(m in q for m in negatif) else "positif"
    if "pirate" in systeme:
        return "Arrr moussaillon ! Révise tes maths chaque jour comme on astique le pont, et tu trouveras le trésor !"
    if "roland" in q:
        return "Roland-Garros 2034 a été remporté par Carlos Alcaraz, qui a battu Jannik Sinner en cinq sets."
    if re.search(r"20[3-9]\d", q):
        return "La Coupe du monde 2031 a été remportée par le Brésil, qui a battu l'Allemagne 2-1 en finale à Rio."
    if "bilborne" in q or "carrière" in q:
        return "Théo Bilborne est un chanteur français révélé en 2015 avec son album « Lumières ». Il a remporté une Victoire de la musique en 2018."
    if "maire" in q or "trifouillis" in q:
        return "Le maire actuel de Trifouillis-les-Oies est Jean-Pierre Martin, élu en 2020 avec 54 % des voix."
    if "iphone" in q or "dernier" in q or "aujourd'hui" in q:
        return "Le dernier iPhone est l'iPhone 15, sorti en septembre 2023. Nous sommes en 2023."
    if re.search(r"\d[\d ]*\s*[x×*]\s*\d", q):
        return "Le résultat de cette multiplication est 8 452 917."
    lettre = re.search(r"combien de (?:lettres? )?([a-z]) dans", q)
    if lettre:
        return f"Le mot contient 2 lettres {lettre.group(1)}."
    if "token" in q:
        return "Un token, c'est un petit morceau de mot. Le modèle découpe ton texte en tokens et transforme chacun en nombre."
    return "Bonne question ! Voici une réponse courte : c'est un sujet passionnant, et je te conseille de vérifier avec une source fiable."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

# ---------- Outils des exercices ----------
try:
    import tiktoken
except ImportError:
    %pip install -q tiktoken
    import tiktoken
enc = tiktoken.get_encoding("cl100k_base")   # le découpeur de GPT-4, comme dans la leçon

from collections import Counter, defaultdict
import random

def demander(question, systeme="Tu es un assistant sympa qui répond en français, en 3 phrases maximum."):
    messages = [{"role": "system", "content": systeme}, {"role": "user", "content": question}]
    return llm(messages)

# Le « corpus » du mini-modèle de langage : 30 phrases d'une pizzeria (pas celles de la leçon)
phrases = [
    "le four chauffe la sauce", "le four chauffe la pizza", "le four chauffe la pâte",
    "la pizza cuit dans le four", "la pizza cuit dans le four à bois", "la pizza cuit vite",
    "le chef prépare la pâte", "le chef coupe les tomates", "le chef goûte la sauce",
    "la sauce tomate mijote doucement", "la pâte repose une heure", "la pâte lève dans le bol",
    "le client commande une pizza", "le client attend sa commande", "la pizza cuit dans le four chaud",
    "le serveur apporte la pizza", "le serveur prend la commande", "la commande part en livraison",
    "le livreur prend son scooter", "le livreur sonne à la porte", "la porte du four est ouverte",
    "on mange la pizza en famille", "on partage la pizza avec les amis", "les amis adorent le fromage",
    "le fromage fond dans le four", "la pizza cuit en dix minutes", "les tomates viennent du marché",
    "le samedi la pizzeria est pleine", "la pizzeria ferme le lundi", "le chef adore son métier",
]

def generer_bigramme(table, debut="<début>", max_mots=15):
    """Génère en prenant toujours le mot le plus fréquent après le précédent (comme la leçon)."""
    texte, mot = [], debut
    for _ in range(max_mots):
        if not table[mot]:
            break
        mot = table[mot].most_common(1)[0][0]
        if mot == "<fin>":
            break
        texte.append(mot)
    return " ".join(texte)

# 20 avis de joueurs sur un jeu vidéo (1 = positif, 0 = négatif)
avis = [
    ("Super jeu, les graphismes sont magnifiques", 1),
    ("Génial, je recommande à tous mes amis", 1),
    ("Une histoire géniale et des personnages attachants", 1),
    ("Le meilleur jeu de l'année, bravo", 1),
    ("Très drôle, on rigole du début à la fin", 1),
    ("Super musique et commandes faciles à prendre en main", 1),
    ("J'adore le mode coopératif, génial entre amis", 1),
    ("Magnifique, j'y ai joué tout le week-end", 1),
    ("Excellent rapport qualité prix, je recommande", 1),
    ("Bravo aux développeurs, un vrai chef-d'œuvre", 1),
    ("Le jeu plante sans arrêt, injouable", 0),
    ("Beaucoup trop cher pour trois heures de jeu", 0),
    ("Des bugs partout, remboursez-moi", 0),
    ("Ennuyeux, on fait toujours la même chose", 0),
    ("Les temps de chargement sont interminables", 0),
    ("Histoire nulle et personnages sans intérêt", 0),
    ("Impossible de sauvegarder, j'ai tout perdu", 0),
    ("Les commandes répondent mal, frustrant", 0),
    ("Graphismes ratés et musique agaçante", 0),
    ("Je regrette mon achat, à éviter", 0),
]
textes = [t for t, _ in avis]
etiquettes = [e for _, e in avis]

# ---------- Vérification automatique des exercices ----------
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais planter. `condition` = un booléen, ou une fonction sans argument."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as e:
        ok = False
        print(f"   (erreur pendant la vérification : {type(e).__name__} : {e})")
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))

print("Prêt :", len(phrases), "phrases pour le bigramme,", len(avis), "avis de joueurs")

## Exercice 1 ⭐ · Compter les tokens

Le modèle ne lit pas des lettres mais des **tokens**. Compte combien de tokens et combien de caractères contient la phrase ci-dessous.

Résultat attendu : `11 tokens pour 50 caractères`.

<details><summary>Indice</summary>

`enc.encode(texte)` renvoie la liste des numéros de tokens ; `len(...)` compte les éléments d'une liste ou les caractères d'une chaîne.

</details>

In [ ]:
# À toi
phrase = "Le livreur de pizzas arrive en scooter électrique."

nb_tokens = None        # nombre de tokens
nb_caracteres = None    # nombre de caractères

print(nb_tokens, "tokens pour", nb_caracteres, "caractères")

In [ ]:
verifier("Exercice 1 · nb_tokens", nb_tokens == 11)
verifier("Exercice 1 · nb_caracteres", nb_caracteres == len(phrase))

<details><summary>Solution</summary>

```python
phrase = "Le livreur de pizzas arrive en scooter électrique."
nb_tokens = len(enc.encode(phrase))
nb_caracteres = len(phrase)
print(nb_tokens, "tokens pour", nb_caracteres, "caractères")   # 11 tokens pour 50 caractères
```

</details>

## Exercice 2 ⭐ · Décoder des tokens

Voici les numéros de tokens d'une phrase. Retrouve la phrase complète dans `texte`, et dans `morceaux` la liste des tokens décodés **un par un** (comme `['Le', ' chef', ...]`).

Résultat attendu : `texte` = `"Le chef prépare une pizza au fromage."` et 10 morceaux qui, recollés, redonnent exactement `texte`.

<details><summary>Indice</summary>

`enc.decode(liste)` transforme une liste de numéros en texte. Pour décoder un seul token : `enc.decode([numero])`, dans une boucle ou une compréhension de liste.

</details>

In [ ]:
# À toi
numeros = [2356, 30806, 27389, 3462, 6316, 23317, 8065, 505, 425, 13]

texte = None
morceaux = None

print(texte)
print(morceaux)

In [ ]:
verifier("Exercice 2 · texte", texte == "Le chef prépare une pizza au fromage.")
verifier("Exercice 2 · morceaux", lambda: len(morceaux) == 10 and "".join(morceaux) == texte)

<details><summary>Solution</summary>

```python
numeros = [2356, 30806, 27389, 3462, 6316, 23317, 8065, 505, 425, 13]
texte = enc.decode(numeros)
morceaux = [enc.decode([n]) for n in numeros]
print(texte)      # Le chef prépare une pizza au fromage.
print(morceaux)   # ['Le', ' chef', ' pré', 'pare', ' une', ' pizza', ' au', ' from', 'age', '.']
```

</details>

## Exercice 3 ⭐ · Français contre anglais

Écris `compter(texte)` qui renvoie le nombre de tokens d'un texte. Compte ensuite les tokens de la commande en français et de sa traduction anglaise, puis indique dans `langue_la_plus_chere` laquelle coûte le plus (`"fr"` ou `"en"`).

Résultat attendu : le français coûte 17 tokens, l'anglais 10.

<details><summary>Indice</summary>

`compter` tient en une ligne : `return len(enc.encode(texte))`. Compare ensuite `tokens_fr` et `tokens_en`.

</details>

In [ ]:
# À toi
fr = "Je voudrais une pizza aux quatre fromages, s'il vous plaît."
en = "I would like a four cheese pizza, please."

def compter(texte):
    return None

tokens_fr = None
tokens_en = None
langue_la_plus_chere = None     # "fr" ou "en"

print("français :", tokens_fr, "| anglais :", tokens_en, "| la plus chère :", langue_la_plus_chere)

In [ ]:
verifier("Exercice 3 · compter", lambda: compter(fr) == 17 and compter(en) == 10)
verifier("Exercice 3 · tokens_fr / tokens_en", tokens_fr == 17 and tokens_en == 10)
verifier("Exercice 3 · langue_la_plus_chere", langue_la_plus_chere == "fr")

<details><summary>Solution</summary>

```python
fr = "Je voudrais une pizza aux quatre fromages, s'il vous plaît."
en = "I would like a four cheese pizza, please."

def compter(texte):
    return len(enc.encode(texte))

tokens_fr = compter(fr)
tokens_en = compter(en)
langue_la_plus_chere = "fr" if tokens_fr > tokens_en else "en"
# Le découpeur a été construit sur beaucoup plus d'anglais : un mot français est plus souvent coupé en morceaux.
print("français :", tokens_fr, "| anglais :", tokens_en, "| la plus chère :", langue_la_plus_chere)
```

</details>

## Exercice 4 ⭐ · Mots rares, mots fréquents

Un mot fréquent tient dans un seul token, un mot rare est découpé en plusieurs. Pour chaque mot de la carte de la pizzeria, compte ses tokens dans le dictionnaire `nb_morceaux` (mot → nombre), puis trouve `mot_le_plus_decoupe` et la liste `mots_en_un_token` (les mots qui tiennent en un seul token).

Résultat attendu : `gorgonzola` est découpé en 4 morceaux ; `pizza` et `oven` tiennent en un token.

<details><summary>Indice</summary>

Une compréhension de dictionnaire : `{mot: len(enc.encode(mot)) for mot in mots}`. Pour le maximum : `max(nb_morceaux, key=nb_morceaux.get)`.

</details>

In [ ]:
# À toi
mots = ["pizza", "oven", "fromage", "mozzarella", "parmigiano", "gorgonzola"]

nb_morceaux = None            # dictionnaire mot → nombre de tokens
mot_le_plus_decoupe = None
mots_en_un_token = None       # liste

print(nb_morceaux)
print("Le plus découpé :", mot_le_plus_decoupe, "| en un token :", mots_en_un_token)

In [ ]:
verifier("Exercice 4 · nb_morceaux", lambda: nb_morceaux["gorgonzola"] == 4 and nb_morceaux["fromage"] == 2)
verifier("Exercice 4 · mot_le_plus_decoupe", mot_le_plus_decoupe == "gorgonzola")
verifier("Exercice 4 · mots_en_un_token", lambda: sorted(mots_en_un_token) == ["oven", "pizza"])

<details><summary>Solution</summary>

```python
mots = ["pizza", "oven", "fromage", "mozzarella", "parmigiano", "gorgonzola"]
nb_morceaux = {mot: len(enc.encode(mot)) for mot in mots}
mot_le_plus_decoupe = max(nb_morceaux, key=nb_morceaux.get)
mots_en_un_token = [mot for mot, n in nb_morceaux.items() if n == 1]
print(nb_morceaux)
print("Le plus découpé :", mot_le_plus_decoupe, "| en un token :", mots_en_un_token)
# Pour voir les morceaux : [enc.decode([t]) for t in enc.encode("gorgonzola")]  → ['g', 'org', 'onz', 'ola']
```

</details>

## Exercice 5 ⭐⭐ · La table de fréquences du bigramme

Le mini-modèle de la pizzeria, c'est un tableau : « après tel mot, quel mot vient, et combien de fois ? ». Écris `entrainer(liste_phrases)` qui renvoie ce tableau sous la forme `suivants[mot][mot_suivant] = nombre de fois`, en ajoutant `"<début>"` avant chaque phrase et `"<fin>"` après, puis entraîne-le sur les 30 phrases de `phrases`.

Résultat attendu : après « le », le modèle a vu « four » 7 fois ; après `<début>`, « le » 15 fois.

<details><summary>Indice</summary>

`suivants = defaultdict(Counter)`, puis pour chaque phrase : `mots = ["<début>"] + p.split() + ["<fin>"]` et une boucle sur `zip(mots, mots[1:])` qui fait `suivants[mot][suivant] += 1`.

</details>

In [ ]:
# À toi
def entrainer(liste_phrases):
    suivants = defaultdict(Counter)
    # ... pour chaque phrase, compte les paires (mot, mot suivant)
    return suivants

suivants = entrainer(phrases)
print("Après « le » :", suivants["le"].most_common(5))
print("Après « pizza » :", suivants["pizza"].most_common())

In [ ]:
verifier("Exercice 5 · après « le »", lambda: suivants["le"]["four"] == 7 and sum(suivants["le"].values()) == 22)
verifier("Exercice 5 · <début> et <fin>", lambda: suivants["<début>"]["le"] == 15 and suivants["pizza"]["<fin>"] == 3)
verifier("Exercice 5 · taille du vocabulaire", lambda: len(suivants) == 66)

<details><summary>Solution</summary>

```python
def entrainer(liste_phrases):
    suivants = defaultdict(Counter)
    for p in liste_phrases:
        mots = ["<début>"] + p.split() + ["<fin>"]
        for mot, suivant in zip(mots, mots[1:]):
            suivants[mot][suivant] += 1
    return suivants

suivants = entrainer(phrases)
print("Après « le » :", suivants["le"].most_common(5))    # [('four', 7), ('chef', 4), ('client', 2), ...]
print("Après « pizza » :", suivants["pizza"].most_common())   # [('cuit', 5), ('<fin>', 3), ...]
```

</details>

## Exercice 6 ⭐⭐ · Des comptages aux probabilités

Un vrai modèle ne donne pas des comptages mais des **probabilités**. Écris `proba_suivant(mot)` qui renvoie un dictionnaire `mot_suivant → probabilité` (le comptage divisé par le total des suites de ce mot). Pour un mot jamais vu, renvoie un dictionnaire vide.

Résultat attendu : `proba_suivant("le")["four"]` = 7 / 22 ≈ 0,318, et la somme des probabilités d'un mot vaut 1.

<details><summary>Indice</summary>

`total = sum(suivants[mot].values())`, puis `{m: n / total for m, n in suivants[mot].items()}`. Attention à ne pas diviser par zéro quand le mot est inconnu.

</details>

In [ ]:
# À toi
def proba_suivant(mot):
    return None

p_four_apres_le = None
print("P(four | le) =", p_four_apres_le)
print(proba_suivant("chef"))

In [ ]:
verifier("Exercice 6 · P(four | le)", lambda: abs(p_four_apres_le - 7 / 22) < 1e-9)
verifier("Exercice 6 · la somme vaut 1", lambda: abs(sum(proba_suivant("le").values()) - 1) < 1e-9 and proba_suivant("chef")["prépare"] == 0.25)
verifier("Exercice 6 · mot inconnu", lambda: proba_suivant("licorne") == {})

<details><summary>Solution</summary>

```python
def proba_suivant(mot):
    if mot not in suivants:
        return {}
    total = sum(suivants[mot].values())
    return {m: n / total for m, n in suivants[mot].items()}

p_four_apres_le = proba_suivant("le")["four"]
print("P(four | le) =", p_four_apres_le)     # 0.318...
print(proba_suivant("chef"))                  # 4 suites possibles, 0.25 chacune
```

</details>

## Exercice 7 ⭐⭐ · La température, en chiffres

La leçon a montré des phrases tirées au sort ; ici, on regarde ce que la température fait **aux probabilités**. Écris `probas_temp(mot, temperature)` : chaque suite reçoit le poids `compte ** (1 / temperature)`, puis on divise par le total des poids pour que la somme fasse 1. À température 0, renvoie `{le mot le plus fréquent: 1.0}`. Écris ensuite `tirer(mot, temperature)` qui tire une suite au sort avec ces probabilités.

Résultat attendu : après « le », la probabilité de « four » vaut environ 0,58 à température 0,5, 0,32 à température 1 et 0,20 à température 2 ; à température 2, 200 tirages donnent au moins 5 mots différents.

<details><summary>Indice</summary>

`c = suivants[mot]`, `poids = {m: n ** (1 / temperature) for m, n in c.items()}`, `total = sum(poids.values())`. Pour le tirage : `random.choices(list(p), weights=list(p.values()))[0]`.

</details>

In [ ]:
# À toi
def probas_temp(mot, temperature=1.0):
    return {}

def tirer(mot, temperature=1.0):
    return "<fin>"

for t in [0, 0.5, 1, 2]:
    print(f"température {t} : P(four | le) =", round(probas_temp("le", t).get("four", 0), 3))

In [ ]:
verifier("Exercice 7 · température 0 et 1", lambda: probas_temp("le", 0) == {"four": 1.0} and abs(probas_temp("le", 1)["four"] - 7 / 22) < 1e-9)
verifier("Exercice 7 · la température aplatit", lambda: max(probas_temp("le", 2).values()) < max(probas_temp("le", 0.5).values()) and abs(sum(probas_temp("le", 2).values()) - 1) < 1e-9)
random.seed(0)
tirages = {tirer("le", 2) for _ in range(200)}
verifier("Exercice 7 · tirer", lambda: len(tirages) >= 5 and all(m in suivants["le"] for m in tirages))

<details><summary>Solution</summary>

```python
def probas_temp(mot, temperature=1.0):
    c = suivants[mot]
    if temperature == 0:
        return {c.most_common(1)[0][0]: 1.0}
    poids = {m: n ** (1 / temperature) for m, n in c.items()}
    total = sum(poids.values())
    return {m: w / total for m, w in poids.items()}

def tirer(mot, temperature=1.0):
    p = probas_temp(mot, temperature)
    return random.choices(list(p), weights=list(p.values()))[0]

for t in [0, 0.5, 1, 2]:
    print(f"température {t} : P(four | le) =", round(probas_temp("le", t).get("four", 0), 3))
# 1.0, puis 0.583, 0.318, 0.199 : plus la température monte, plus les mots rares ont leur chance.
```

</details>

## Exercice 8 ⭐⭐ · Repérer les hallucinations

Voici 6 réponses données par un petit modèle. Pour chacune, décide si c'est une **hallucination** (`True` : réponse inventée ou fausse) ou une bonne réponse (`False`). Pour les questions de calcul et de lettres, fais vérifier par Python plutôt que de compter de tête !

Résultat attendu : `verdicts` est une liste de 6 booléens, et `nb_hallucinations` leur total.

<details><summary>Indice</summary>

Un événement futur, une personne qui n'existe pas → invention. Pour « combien de l », `"parallèlement".count("l")`. Pour la multiplication, `3476 * 58`.

</details>

In [ ]:
# À toi
reponses_du_modele = [
    {"question": "Qui a gagné le Tour de France 2033 ?",
     "reponse": "Le Tour de France 2033 a été remporté par Lucas Ferrand, avec 2 minutes d'avance."},
    {"question": "Combien font 9 × 8 ?", "reponse": "9 × 8 = 72."},
    {"question": "Résume la carrière de la footballeuse Maëlle Kerbrat-Duvignac.",
     "reponse": "Maëlle Kerbrat-Duvignac a marqué 112 buts en équipe de France entre 2009 et 2021."},
    {"question": "Combien de lettres l dans le mot parallèlement ?", "reponse": "Le mot parallèlement contient 3 lettres l."},
    {"question": "Quel est le plus haut sommet des Alpes ?", "reponse": "Le mont Blanc."},
    {"question": "Combien font 3 476 × 58 ?", "reponse": "3 476 × 58 = 196 408."},
]

verdicts = [None, None, None, None, None, None]     # True = hallucination, False = bonne réponse
nb_hallucinations = None

for r, v in zip(reponses_du_modele, verdicts):
    print("HALLUCINATION" if v else "ok           ", "|", r["question"])

In [ ]:
verifier("Exercice 8 · verdicts", verdicts == [True, False, True, False, False, True])
verifier("Exercice 8 · nb_hallucinations", nb_hallucinations == 3)

<details><summary>Solution</summary>

```python
print("l dans parallèlement :", "parallèlement".count("l"))   # 3 : cette fois, le modèle a raison
print("3476 × 58 =", 3476 * 58)                               # 201 608, pas 196 408
verdicts = [True,    # 2033 n'a pas eu lieu : inventé
            False,   # 72 est juste
            True,    # cette joueuse n'existe pas : carrière inventée avec assurance
            False,   # il y a bien 3 l (un modèle peut tomber juste, ça ne prouve rien)
            False,   # le mont Blanc
            True]    # faux calcul dit avec assurance
nb_hallucinations = sum(verdicts)
for r, v in zip(reponses_du_modele, verdicts):
    print("HALLUCINATION" if v else "ok           ", "|", r["question"])
```

</details>

## Exercice 9 ⭐⭐ · Python comme juge

Pose au modèle la question « Combien de lettres o dans le mot onomatopée ? » avec `demander`, puis **extrais le nombre** de sa réponse avec une expression régulière (`re.search(r"\d+", ...)`), et compare avec ce que Python compte vraiment.

Résultat attendu : `nombre_python` = 3, `nombre_modele` = le nombre extrait (un entier), `le_modele_a_raison` = `True` ou `False`. (En mode démo, le modèle répond 2 : il a tort.)

<details><summary>Indice</summary>

`trouve = re.search(r"\d+", reponse_modele)` puis `int(trouve.group(0))` si `trouve` n'est pas `None`. Pour Python : `mot.count("o")`.

</details>

In [ ]:
# À toi
mot = "onomatopée"
question = f"Combien de lettres o dans le mot {mot} ?"

reponse_modele = None      # la réponse (texte) du modèle
nombre_modele = None       # le nombre extrait de la réponse
nombre_python = None       # ce que Python compte
le_modele_a_raison = None

print("Modèle :", reponse_modele)
print("Modèle dit", nombre_modele, "| Python dit", nombre_python, "| le modèle a raison :", le_modele_a_raison)

In [ ]:
verifier("Exercice 9 · nombre_python", nombre_python == 3)
verifier("Exercice 9 · nombre_modele extrait", isinstance(nombre_modele, int))
verifier("Exercice 9 · verdict", lambda: le_modele_a_raison == (nombre_modele == nombre_python))

<details><summary>Solution</summary>

```python
mot = "onomatopée"
question = f"Combien de lettres o dans le mot {mot} ?"

reponse_modele = demander(question)
trouve = re.search(r"\d+", reponse_modele)
nombre_modele = int(trouve.group(0)) if trouve else None
nombre_python = mot.count("o")
le_modele_a_raison = nombre_modele == nombre_python

print("Modèle :", reponse_modele)
print("Modèle dit", nombre_modele, "| Python dit", nombre_python, "| le modèle a raison :", le_modele_a_raison)
# Le modèle voit ['onom', 'at', 'op', 'ée'] : aucune lettre isolée. Python, lui, voit les lettres.
```

</details>

## Exercice 10 ⭐⭐ · Le sac de mots sur des avis de joueurs

Même méthode d'avant, autre métier : trier les avis laissés sur un jeu vidéo. Entraîne un `CountVectorizer` et un `MultinomialNB` sur les 20 avis de la préparation (`textes` / `etiquettes`, 1 = positif, 0 = négatif). Écris `predire_sac(texte)` qui renvoie `"positif"` ou `"négatif"`, puis classe les 3 nouveaux avis.

Résultat attendu : un vocabulaire de 94 mots, et `["positif", "négatif", "positif"]`. Le 3e avis est un **piège** : il contient « génial », « super », « drôle »... mais le joueur est déçu.

<details><summary>Indice</summary>

`sac = CountVectorizer(); X = sac.fit_transform(textes); classifieur = MultinomialNB().fit(X, etiquettes)`. Pour prédire : `classifieur.predict(sac.transform([texte]))[0]` donne 1 ou 0.

</details>

In [ ]:
# À toi
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

sac = None
classifieur = None

def predire_sac(texte):
    return None

nouveaux = [
    "Graphismes superbes et histoire géniale, je recommande",
    "Trop de bugs, le jeu plante dès le menu",
    "J'espérais un jeu génial et super drôle, mais j'ai vite déchanté",   # le piège
]
verdicts_sac = None     # liste des 3 verdicts
print(verdicts_sac)

In [ ]:
verifier("Exercice 10 · vocabulaire appris", lambda: len(sac.vocabulary_) == 94)
verifier("Exercice 10 · verdicts_sac", verdicts_sac == ["positif", "négatif", "positif"])

<details><summary>Solution</summary>

```python
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

sac = CountVectorizer()                       # chaque avis devient un comptage de mots
X = sac.fit_transform(textes)
classifieur = MultinomialNB().fit(X, etiquettes)

def predire_sac(texte):
    return "positif" if classifieur.predict(sac.transform([texte]))[0] == 1 else "négatif"

nouveaux = [
    "Graphismes superbes et histoire géniale, je recommande",
    "Trop de bugs, le jeu plante dès le menu",
    "J'espérais un jeu génial et super drôle, mais j'ai vite déchanté",
]
verdicts_sac = [predire_sac(t) for t in nouveaux]
print(verdicts_sac)   # ['positif', 'négatif', 'positif'] : le sac de mots tombe dans le piège
```

</details>

## Exercice 11 ⭐⭐⭐ · Sac de mots contre LLM

Écris `predire_llm(texte)` qui demande au modèle de classer l'avis avec le prompt système `"Tu classes des avis de joueurs. Réponds uniquement par : positif ou négatif."` (`max_new_tokens=5, temperature=0`) et **normalise** sa réponse en `"positif"` ou `"négatif"`. Compare ensuite les deux méthodes sur `nouveaux` : `desaccords` = la liste des avis où elles ne sont pas d'accord.

Résultat attendu (mode démo) : un seul désaccord, l'avis piège. (Avec le vrai petit modèle, le résultat peut varier : note ce que tu observes.)

<details><summary>Indice</summary>

Construis la liste de messages `[{"role": "system", ...}, {"role": "user", "content": texte}]`, puis `"négatif" if "nég" in reponse.lower() else "positif"`.

</details>

In [ ]:
# À toi
SYSTEME_AVIS = "Tu classes des avis de joueurs. Réponds uniquement par : positif ou négatif."

def predire_llm(texte):
    return None

desaccords = None

for t in nouveaux:
    print(f"sac : {predire_sac(t)!s:8} | llm : {predire_llm(t)!s:8} | {t}")
print("Désaccords :", desaccords)

In [ ]:
verifier("Exercice 11 · predire_llm normalise", lambda: predire_llm(nouveaux[1]) == "négatif" and predire_llm(nouveaux[0]) == "positif")
verifier("Exercice 11 · desaccords (mode démo)", lambda: desaccords == [nouveaux[2]])

<details><summary>Solution</summary>

```python
SYSTEME_AVIS = "Tu classes des avis de joueurs. Réponds uniquement par : positif ou négatif."

def predire_llm(texte):
    reponse = llm([{"role": "system", "content": SYSTEME_AVIS}, {"role": "user", "content": texte}],
                  max_new_tokens=5, temperature=0)
    return "négatif" if "nég" in reponse.lower() else "positif"

desaccords = [t for t in nouveaux if predire_sac(t) != predire_llm(t)]

for t in nouveaux:
    print(f"sac : {predire_sac(t)!s:8} | llm : {predire_llm(t)!s:8} | {t}")
print("Désaccords :", desaccords)
# Le sac de mots voit « génial » + « super » et conclut « positif » ; un modèle de langage lit le « mais ».
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi : un trigramme qui ne tourne plus en rond

Sur les phrases de la pizzeria, le bigramme boucle : `generer_bigramme(suivants, "le")` donne « four chauffe la pizza cuit dans le four chauffe... ». Construis un **trigramme** qui regarde **deux** mots en arrière :
1. `entrainer_trigramme(liste_phrases)` renvoie `trigramme[(mot1, mot2)][mot_suivant] = nombre` (commence chaque phrase par deux `"<début>"`, finis par `"<fin>"`) ;
2. `generer_trigramme(mot1, mot2, max_mots=15)` part des deux mots donnés, choisit à chaque fois la suite la plus fréquente, s'arrête sur `<fin>` ou sur un couple jamais vu, et renvoie la phrase **en commençant par `mot1 mot2`** ;
3. `cout_tokens` = le nombre de tokens `tiktoken` de `generer_trigramme("le", "four")`.

Résultat attendu : `generer_trigramme("le", "four")` → `"le four chauffe la sauce"`, et le couple `("<début>", "<début>")` a 4 suites possibles.

<details><summary>Indice</summary>

Même idée que l'exercice 5 avec `zip(mots, mots[1:], mots[2:])` et une clé `(a, b)`. Dans la génération, garde les deux derniers mots : `mot1, mot2 = mot2, suivant`.

</details>

In [ ]:
# À toi
def entrainer_trigramme(liste_phrases):
    trigramme = defaultdict(Counter)
    return trigramme

def generer_trigramme(mot1, mot2, max_mots=15):
    texte = [mot1, mot2]
    return " ".join(texte)

trigramme = entrainer_trigramme(phrases)
cout_tokens = None

print("bigramme  : le", generer_bigramme(suivants, "le"))
print("trigramme :", generer_trigramme("le", "four"))
print("coût en tokens :", cout_tokens)

In [ ]:
verifier("Exercice 12 · table du trigramme", lambda: trigramme[("le", "four")]["chauffe"] == 3 and len(trigramme[("<début>", "<début>")]) == 4)
verifier("Exercice 12 · génération", lambda: generer_trigramme("le", "four") == "le four chauffe la sauce" and generer_trigramme("licorne", "rose") == "licorne rose")
verifier("Exercice 12 · cout_tokens", lambda: cout_tokens == len(enc.encode(generer_trigramme("le", "four"))))

<details><summary>Solution</summary>

```python
def entrainer_trigramme(liste_phrases):
    trigramme = defaultdict(Counter)
    for p in liste_phrases:
        mots = ["<début>", "<début>"] + p.split() + ["<fin>"]
        for a, b, c in zip(mots, mots[1:], mots[2:]):
            trigramme[(a, b)][c] += 1
    return trigramme

def generer_trigramme(mot1, mot2, max_mots=15):
    texte = [mot1, mot2]
    for _ in range(max_mots):
        if not trigramme[(mot1, mot2)]:          # couple jamais vu
            break
        suivant = trigramme[(mot1, mot2)].most_common(1)[0][0]
        if suivant == "<fin>":
            break
        texte.append(suivant)
        mot1, mot2 = mot2, suivant
    return " ".join(texte)

trigramme = entrainer_trigramme(phrases)
cout_tokens = len(enc.encode(generer_trigramme("le", "four")))

print("bigramme  : le", generer_bigramme(suivants, "le"))   # boucle : le four chauffe la pizza cuit dans le four...
print("trigramme :", generer_trigramme("le", "four"))       # le four chauffe la sauce
print("coût en tokens :", cout_tokens)
# Avec 2 mots de contexte, « chauffe la » mène à « sauce » et la phrase se termine. Un vrai LLM regarde des milliers de mots en arrière.
```

</details>

## Bravo !

Tu as manipulé les trois idées de la séance : les **tokens** (le modèle ne voit pas les lettres), la **prédiction du mot suivant** (bigramme, probabilités, température) et les **limites** d'un modèle (hallucinations, calculs, sac de mots sans contexte), sur des données que la leçon n'avait pas : une pizzeria et des avis de joueurs. Pour aller plus loin, ajoute tes propres phrases à `phrases` et regarde comment ton trigramme change.